In [ ]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Pair Distribution Function: NaCl, XRD

This example demonstrates a pair distribution function (PDF) analysis
of NaCl, based on data collected from an X-ray powder diffraction
experiment.

The dataset is taken from:
https://github.com/diffpy/add2019-diffpy-cmi/tree/master

## Import Library

In [ ]:
import easydiffraction as ed

## Create Project

In [ ]:
project = ed.Project()

## Set Plotting Engine

In [ ]:
# Keep the auto-selected engine. Alternatively, you can uncomment the
# line below to explicitly set the engine to the required one.
# project.plotter.engine = 'plotly'

In [ ]:
# Set global plot range for plots
project.plotter.x_min = 2.0
project.plotter.x_max = 30.0

## Add Structure

In [ ]:
project.structures.create(name='nacl')

In [ ]:
project.structures['nacl'].space_group.name_h_m = 'F m -3 m'
project.structures['nacl'].space_group.it_coordinate_system_code = '1'
project.structures['nacl'].cell.length_a = 5.62
project.structures['nacl'].atom_sites.create(
    label='Na',
    type_symbol='Na',
    fract_x=0,
    fract_y=0,
    fract_z=0,
    wyckoff_letter='a',
    b_iso=1.0,
)
project.structures['nacl'].atom_sites.create(
    label='Cl',
    type_symbol='Cl',
    fract_x=0.5,
    fract_y=0.5,
    fract_z=0.5,
    wyckoff_letter='b',
    b_iso=1.0,
)

## Add Experiment

In [ ]:
data_path = ed.download_data(id=4, destination='data')

In [ ]:
project.experiments.add_from_data_path(
    name='xray_pdf',
    data_path=data_path,
    sample_form='powder',
    beam_mode='constant wavelength',
    radiation_probe='xray',
    scattering_type='total',
)

In [ ]:
project.experiments['xray_pdf'].show_supported_peak_profile_types()

In [ ]:
project.experiments['xray_pdf'].show_current_peak_profile_type()

In [ ]:
project.experiments['xray_pdf'].peak_profile_type = 'gaussian-damped-sinc'

In [ ]:
project.experiments['xray_pdf'].peak.damp_q = 0.03
project.experiments['xray_pdf'].peak.broad_q = 0
project.experiments['xray_pdf'].peak.cutoff_q = 21
project.experiments['xray_pdf'].peak.sharp_delta_1 = 0
project.experiments['xray_pdf'].peak.sharp_delta_2 = 5
project.experiments['xray_pdf'].peak.damp_particle_diameter = 0

In [ ]:
project.experiments['xray_pdf'].linked_phases.create(id='nacl', scale=0.5)

## Select Fitting Parameters

In [ ]:
project.structures['nacl'].cell.length_a.free = True
project.structures['nacl'].atom_sites['Na'].b_iso.free = True
project.structures['nacl'].atom_sites['Cl'].b_iso.free = True

In [ ]:
project.experiments['xray_pdf'].linked_phases['nacl'].scale.free = True
project.experiments['xray_pdf'].peak.damp_q.free = True
project.experiments['xray_pdf'].peak.sharp_delta_2.free = True

## Run Fitting

In [ ]:
project.analysis.fit()
project.analysis.display.fit_results()

## Plot Measured vs Calculated

In [ ]:
project.plotter.plot_meas_vs_calc(expt_name='xray_pdf')